In [1]:
import os
import numpy as np
import pandas as pd
import mne
import neurokit2 as nk
from scipy.signal import resample
from scipy.signal import welch
import matplotlib.pyplot as plt
import os, neurokit2 as nk


In [2]:
F_M_Path = r"F:\data\participants.tsv"
F_M = pd.read_csv(F_M_Path, sep="\t")
F_M

,participant_id,sex
0,sub-001,f
1,sub-002,m
2,sub-003,m
3,sub-004,m
4,sub-005,f
...,...,...
120,sub-121,m
121,sub-122,m
122,sub-123,m
123,sub-124,m


### Read ECG

In [3]:
def Read_ECG(subject_id, base_folder=r"F:\data"):
    subject_folder = os.path.join(base_folder, subject_id)
    ecg_data = {}

    for ses in os.listdir(subject_folder):
        ses_folder = os.path.join(subject_folder, ses)
        if os.path.isdir(ses_folder) and ses.startswith("ses-"):
            ecg_folder = os.path.join(ses_folder, "ecg")
            if os.path.exists(ecg_folder):
                for file in os.listdir(ecg_folder):
                    if file.endswith(".edf"):
                        file_path = os.path.join(ecg_folder, file)
                        raw = mne.io.read_raw_edf(file_path, preload=True)
                        raw.pick_channels(['ECG SD'])
                        raw.rename_channels({'ECG SD': 'ECG'})
                        ecg_data[f"{ses}_{file}"] = raw
                        print(f"Loaded {file} from {ses}")
    
    return ecg_data

In [4]:
def ECG_Dataframe(ecg_dict, subject_id, participants_file):
    participants_df = pd.read_csv(participants_file, sep='\t')
    subject_num = subject_id.replace("sub-", "")
    match = participants_df[
        participants_df['participant_id'].isin([subject_id, subject_num])
    ]

    sex = match['sex'].values[0] if not match.empty else "Unknown"

    data_list = []
    for key, raw in ecg_dict.items():
        ses_file_split = key.split("_")
        ses = ses_file_split[0]

        run = None
        for part in ses_file_split:
            if part.startswith("run-"):
                run = part.replace("run-", "")
                break

        ecg_signal, times = raw.get_data(return_times=True)
        ecg_signal = raw.get_data()[0]

        n_samples = len(ecg_signal)
        sfreq = raw.info['sfreq']

        temp_dict = {
            'subject': subject_num,
            'run': run,
            'sfreq': sfreq,
            'n_samples': n_samples,
            'signal': ecg_signal,
            'times': times,
            'sex': sex
        }
        data_list.append(temp_dict)

    ecg_df = pd.DataFrame(data_list)
    print(f"DataFrame summary created with {len(ecg_df)} rows (files) for subject {subject_id} ({sex}).")
    return ecg_df


### Read ACC

In [ ]:
def Read_ACC(subject_id, base_folder=r"F:\data"):
   
    subject_folder = os.path.join(base_folder, subject_id)
    acc_data = {}

    for ses in os.listdir(subject_folder):
        ses_folder = os.path.join(subject_folder, ses)
        if os.path.isdir(ses_folder) and ses.startswith("ses-"):
            acc_folder = os.path.join(ses_folder, "mov")  
            if os.path.exists(acc_folder):
                for file in os.listdir(acc_folder):
                    if file.endswith(".edf"):
                        file_path = os.path.join(acc_folder, file)
                        raw = mne.io.read_raw_edf(file_path, preload=True)
                        
                        acc_channels = [ch for ch in raw.ch_names if 'ACC X' in ch or 'ACC Y' in ch or 'ACC Z' in ch]
                        if len(acc_channels) < 3:
                            print(f"Warning: Not all ACC channels found in {file}. Found: {acc_channels}")
                        else:
                            raw.pick_channels(acc_channels)
                            acc_data[f"{ses}_{file}"] = raw
                            print(f"Loaded {file} from {ses} with channels: {acc_channels}")
    
    return acc_data

In [ ]:
def ACC_Dataframe(acc_dict, subject_id, participants_file):

    participants_df = pd.read_csv(participants_file, sep='\t')
    subject_num = subject_id.replace("sub-", "")
    match = participants_df[
        participants_df['participant_id'].isin([subject_id, subject_num])]

    sex = match['sex'].values[0] if not match.empty else "Unknown"

    data_list = []
    for key, raw in acc_dict.items():
        ses_file_split = key.split("_")
        ses = ses_file_split[0]

        run = None
        for part in ses_file_split:
            if part.startswith("run-"):
                run = part.replace("run-", "")
                break

        acc_signal, times = raw.get_data(return_times=True)  
        acc_X = acc_signal[0]
        acc_Y = acc_signal[1]
        acc_Z = acc_signal[2]

        acc_magnitude = np.sqrt(acc_X**2 + acc_Y**2 + acc_Z**2)

        n_samples = acc_signal.shape[1]
        sfreq = raw.info['sfreq']

        temp_dict = {
            'subject': subject_num,
            'run': run,
            'sfreq': sfreq,
            'n_samples': n_samples,
            'signal': acc_signal,            
            'ACC_X': acc_X,
            'ACC_Y': acc_Y,
            'ACC_Z': acc_Z,
            'acc_magnitude': acc_magnitude,
            'times': times,
            'sex': sex
        }
        data_list.append(temp_dict)

    acc_df = pd.DataFrame(data_list)
    print(f"DataFrame summary created with {len(acc_df)} rows (files) for subject {subject_id} ({sex}).")
    return acc_df

### Read annotation

In [5]:
def Read_annotation(subject_id, base_folder=r"F:\data"):
    subject_folder = os.path.join(base_folder, subject_id)
    ann_dict = {}

    for ses in os.listdir(subject_folder):
        ses_folder = os.path.join(subject_folder, ses)
        if os.path.isdir(ses_folder) and ses.startswith("ses-"):
            eeg_folder = os.path.join(ses_folder, "eeg")
            if os.path.exists(eeg_folder):
                for file in os.listdir(eeg_folder):
                    if file.endswith(".tsv"):
                        file_path = os.path.join(eeg_folder, file)
                        ann = pd.read_csv(file_path, sep="\t")
                        ann_dict[f"{ses}_{file}"] = ann
                        print(f"Loaded {file} from {ses}")
    return ann_dict

In [6]:
def Annotation_Dataframe(ann_dict, subject_id):
    data_list = []
    subject_num = subject_id.replace("sub-", "")

    for key, ann in ann_dict.items():
        ses_file_split = key.split("_")
        ses = ses_file_split[0]

        run = None
        for part in ses_file_split:
            if part.startswith("run-"):
                run = part.replace("run-", "")
                break

        if all(col in ann.columns for col in ['onset', 'duration', 'eventType']):
            temp_df = pd.DataFrame({
                'subject': subject_num,
                'run': [run] * len(ann),
                'onset': ann['onset'].values,
                'duration': ann['duration'].values,
                'eventType': ann['eventType'].values,
            })
            data_list.append(temp_df)

    df_ann = pd.concat(data_list, ignore_index=True)
    print(f"Total annotations: {len(df_ann)}")
    return df_ann

### Merge

In [7]:
def merge_edf_events(edf_df, events_df):

    merged_df = pd.merge(
        edf_df,
        events_df,
        on=["subject", "run"],
        how="left"
    )

    return merged_df

### Clean

In [8]:
def remove_unwanted_events(df):
    df_clean = df[~df["eventType"].str.contains("impd|bckg", case=False, na=False)]
    return df_clean

### Processing ECG

In [9]:
def preprocess_ecg_df(df):

    for idx, row in df.iterrows():
        signal = row['signal']
        sfreq = row['sfreq']

        clean_signal = nk.ecg_clean(signal, sampling_rate=sfreq, method="neurokit")

        signal_min = np.min(clean_signal)
        signal_max = np.max(clean_signal)
        den = signal_max - signal_min

        if den != 0:
            signal_norm = (clean_signal - signal_min) / den
        else:
            signal_norm = clean_signal

        df.at[idx, 'signal'] = signal_norm

    print("ECG preprocessing complete: bandpass + notch + safe normalize (0-1)")
    return df

### Processing ACC

In [ ]:
ACC_FS = 25
LOWCUT = 0.5          
HIGHCUT = 10.0        
MOTION_LPF = 10.0

In [ ]:
def preprocess_acc_df(df):
   
    for idx, row in df.iterrows():
        acc_signal = np.array(row['signal']) 
        sfreq = row['sfreq']
        clean_acc = np.zeros_like(acc_signal)

        for i in range(3):
            temp = nk.signal_filter(acc_signal[i], sampling_rate=sfreq, lowcut=LOWCUT, method="butterworth")
            temp = nk.signal_filter(temp, sampling_rate=sfreq, highcut=HIGHCUT, method="butterworth")
            clean_acc[i] = temp

        signal_min = clean_acc.min(axis=1, keepdims=True)
        signal_max = clean_acc.max(axis=1, keepdims=True)
        den = signal_max - signal_min
        den[den == 0] = 1
        norm_acc = (clean_acc - signal_min) / den

        magnitude = np.sqrt(np.sum(norm_acc**2, axis=0))

        df.at[idx, 'signal'] = norm_acc
        df.at[idx, 'ACC_X'] = norm_acc[0]
        df.at[idx, 'ACC_Y'] = norm_acc[1]
        df.at[idx, 'ACC_Z'] = norm_acc[2]
        df.at[idx, 'acc_magnitude'] = magnitude

    print("ACC preprocessing complete: bandpass 0.5-10Hz + normalize + magnitude")
    return df

### Extend time

In [10]:
def extend_seizure_times(df, pre=30, post=30):

    df = df.copy()
    
    df['onset_extended'] = df['onset'] - pre
    df['onset_extended'] = df['onset_extended'].apply(lambda x: max(x, 0)) 
    
    df['duration_extended'] = df['duration'] + pre + post

    print(f"Seizure times extended: +{pre}s before, +{post}s after.")
    return df

### Segment

In [11]:
PRE_SEC = 300
POST_SEC = 300

In [12]:
def segment_ecg_windows(df,
                                 window_sec=60,
                                 step_sec=10,
                                 label_threshold=0.8,
                                 use_extended=True):

    segments = []
    window_id = 0

    for (subject, run), g in df.groupby(["subject", "run"], sort=False):
        g = g.reset_index(drop=True)
        sfreq = float(g["sfreq"].iloc[0])

        for idx, row in g.iterrows():
            ecg_signal = row["signal"]
            times = row["times"]

            labels_seq = np.array(["interictal"] * len(ecg_signal))

            if use_extended and 'onset_extended' in row and 'duration_extended' in row:
                ictal_start = row["onset_extended"]
                ictal_end = ictal_start + row["duration_extended"]
            else:
                ictal_start = row.get("onset", None)
                ictal_end = ictal_start + row.get("duration", 0) if ictal_start is not None else None

            if ictal_start is not None and ictal_end is not None:
                for i, t in enumerate(times):
                    if ictal_start <= t < ictal_end:
                        labels_seq[i] = "ictal"
                    elif ictal_start - PRE_SEC <= t < ictal_start:
                        labels_seq[i] = "preictal"
                    elif ictal_end <= t < ictal_end + POST_SEC:
                        labels_seq[i] = "postictal"

            win_len = int(window_sec * sfreq)
            step_len = int(step_sec * sfreq)

            for start_idx in range(0, len(ecg_signal) - win_len + 1, step_len):
                end_idx = start_idx + win_len
                w_sig = ecg_signal[start_idx:end_idx]
                w_times = times[start_idx:end_idx]
                w_labels = labels_seq[start_idx:end_idx]

                unique, counts = np.unique(w_labels, return_counts=True)
                ratio = dict(zip(unique, counts / len(w_labels)))

                if ratio.get("preictal", 0) >= label_threshold:
                    window_label = "preictal"
                elif ratio.get("ictal", 0) >= label_threshold:
                    window_label = "ictal"
                elif ratio.get("postictal", 0) >= label_threshold:
                    window_label = "postictal"
                else:
                    window_label = "normal"

                segments.append({
                    "window_id": window_id,
                    "subject": subject,
                    "run": run,
                    "start_time": w_times[0],
                    "end_time": w_times[-1],
                    "signal": w_sig,
                    "times": w_times,
                    'eventType': row['eventType'],
                    "label": window_label,
                    "preictal_ratio": ratio.get("preictal", 0.0),
                    "ictal_ratio": ratio.get("ictal", 0.0),
                    "postictal_ratio": ratio.get("postictal", 0.0),
                    "window_length": len(w_sig),
                    "sfreq": sfreq,
                    "sex": row.get("sex", "Unknown")
                    
                })
                window_id += 1

    seg_df = pd.DataFrame(segments)
    print(f"Segmentation complete: {len(seg_df)} windows created after extension.")
    return seg_df

In [ ]:
 def segment_acc_windows(df,
                        window_sec=60,
                        step_sec=10,
                        label_threshold=0.8,
                        use_extended=True):

    segments = []
    window_id = 0

    for (subject, run), g in df.groupby(["subject", "run"], sort=False):
        g = g.reset_index(drop=True)
        sfreq = float(g["sfreq"].iloc[0])

        for idx, row in g.iterrows():
            acc_signal = row["signal"]     
            times = row["times"]

            labels_seq = np.array(["interictal"] * acc_signal.shape[1])

            if use_extended and 'onset_extended' in row and 'duration_extended' in row:
                ictal_start = row["onset_extended"]
                ictal_end = ictal_start + row["duration_extended"]
            else:
                ictal_start = row.get("onset", None)
                ictal_end = ictal_start + row.get("duration", 0) if ictal_start is not None else None

            if ictal_start is not None and ictal_end is not None:
                for i, t in enumerate(times):
                    if ictal_start <= t < ictal_end:
                        labels_seq[i] = "ictal"
                    elif ictal_start - PRE_SEC <= t < ictal_start:
                        labels_seq[i] = "preictal"
                    elif ictal_end <= t < ictal_end + POST_SEC:
                        labels_seq[i] = "postictal"

            win_len = int(window_sec * sfreq)
            step_len = int(step_sec * sfreq)

            for start_idx in range(0, acc_signal.shape[1] - win_len + 1, step_len):
                end_idx = start_idx + win_len
                w_sig = acc_signal[:, start_idx:end_idx]
                w_times = times[start_idx:end_idx]
                w_labels = labels_seq[start_idx:end_idx]

                unique, counts = np.unique(w_labels, return_counts=True)
                ratio = dict(zip(unique, counts / len(w_labels)))

                if ratio.get("preictal", 0) >= label_threshold:
                    window_label = "preictal"
                elif ratio.get("ictal", 0) >= label_threshold:
                    window_label = "ictal"
                elif ratio.get("postictal", 0) >= label_threshold:
                    window_label = "postictal"
                else:
                    window_label = "normal"

                segments.append({
                    "window_id": window_id,
                    "subject": subject,
                    "run": run,
                    "start_time": w_times[0],
                    "end_time": w_times[-1],
                    "ACC_X": w_sig[0],
                    "ACC_Y": w_sig[1],
                    "ACC_Z": w_sig[2],
                    "acc_magnitude": np.sqrt(np.sum(w_sig**2, axis=0)),
                    "times": w_times,
                    "label": window_label,
                    "preictal_ratio": ratio.get("preictal", 0.0),
                    "ictal_ratio": ratio.get("ictal", 0.0),
                    "postictal_ratio": ratio.get("postictal", 0.0),
                    "window_length": w_sig.shape[1],
                    "sfreq": sfreq,
                    "sex": row.get("sex", "Unknown")
                })
                window_id += 1

    seg_df = pd.DataFrame(segments)
    print(f"ACC Segmentation complete: {len(seg_df)} windows created with 4 labels.")
    return seg_df

### Feature Extraction ECG

#### Detect R_Peaks & RR_Interval & HR_Series

In [13]:
def Detect_R_peaks(signal, times, show_plot=False):
    try:
        signal = np.array(signal, dtype=float)
        times = np.array(times, dtype=float)

        if len(signal) < 2 or len(times) < 2:
            return np.array([]), np.zeros(len(signal)), np.array([]), "window too short"
        if np.any(np.isnan(signal)) or np.any(np.isnan(times)):
            return np.array([]), np.zeros(len(signal)), np.array([]), "NaN detected"

        sfreq = 1 / np.mean(np.diff(times))
        clean_signal = nk.ecg_clean(signal, sampling_rate=sfreq, method="neurokit")
        signals, info = nk.ecg_process(clean_signal, sampling_rate=sfreq)
        R_peaks_idx = info["ECG_R_Peaks"]

        if len(R_peaks_idx) < 2:
            return np.array([]), np.zeros(len(signal)), np.array([]), "less than 2 R-peaks"

        R_peaks_times = times[R_peaks_idx]
        RR_intervals_s = np.diff(R_peaks_times)  # seconds

        # Convert RR_intervals to ms for HRV calculation
        RR_intervals_ms = RR_intervals_s * 1000.0

        HR_series_peaks = 60 / RR_intervals_s  # bpm
        HR_series = np.interp(times[1:], R_peaks_times[1:], HR_series_peaks)
        HR_series = np.insert(HR_series, 0, HR_series[0])

        if show_plot:
            nk.events_plot(R_peaks_idx, signal)

        return R_peaks_times, HR_series, RR_intervals_ms, "success"

    except Exception as e:
        return np.array([]), np.zeros(len(signal)), np.array([]), f"error: {e}"

In [ ]:
def Apply_R_peak_Detection(df, show_plot=False):
    results = []
    for i, row in df.iterrows():
        print(f"Processing subject {row['subject']} | run {row['run']} | window {i}")
        signal = row['signal']
        times = row['times']

        R_peaks, HR_series, RR_intervals_ms, status = Detect_R_peaks(signal, times, show_plot=show_plot)

        results.append({
            'subject': row['subject'],
            'run': row['run'],
            'sex': row['sex'],
            'start_time': row['start_time'],
            'end_time': row['end_time'],
            'eventtype': row['eventType'],
            'label': row['label'],
            'signal': signal,
            'times': times,
            'R_peaks': R_peaks if status=="success" else np.array([]),
            'RR_intervals_ms': RR_intervals_ms if status=="success" else np.array([]),
            'HR_series': HR_series if status=="success" else np.zeros(len(signal)),
            'status': status
        })

    df_result = pd.DataFrame(results)
    print(f"Finished R-peak detection for {len(df_result)} windows.")
    return df_result

#### Feature Time Domain General

In [15]:
def Compute_HR_summary(df_hr):
    summary_list = []
    for idx, row in df_hr.iterrows():
        hr_series = np.array(row.get('HR_series', np.zeros_like(row['signal'])))
        if len(hr_series) > 0:
            summary = {
                'meanHR': np.mean(hr_series),
                'stdHR': np.std(hr_series),
                'minHR': np.min(hr_series),
                'maxHR': np.max(hr_series)
            }
        else:
            summary = {'meanHR': np.nan, 'stdHR': np.nan, 'minHR': np.nan, 'maxHR': np.nan}
        summary_list.append(summary)

    df_summary = pd.DataFrame(summary_list)
    df_hr_summary = pd.concat([df_hr.reset_index(drop=True), df_summary], axis=1)
    print(f"HR summary computed for {len(df_hr_summary)} windows.")
    return df_hr_summary

In [16]:
def Compute_Baseline_HR(df_hr, pre_ictal_only=False):
    baseline_dict = {}
    subjects = df_hr['subject'].unique()
    for subj in subjects:
        subj_data = df_hr[df_hr['subject'] == subj]
        if pre_ictal_only:
            subj_clean = subj_data[subj_data['label'] == 'normal']
        else:
            subj_clean = subj_data[~subj_data['label'].isin(['ictal', 'pre-ictal', 'post-ictal'])]
        subj_clean = subj_clean.head(int(600 / 60))  
        baseline_HR = np.nanmean(subj_clean['meanHR'])
        baseline_dict[subj] = baseline_HR
    return baseline_dict

In [17]:
def Compute_Delta_HR(df_hr, baseline_dict):
    delta_list = []
    for _, row in df_hr.iterrows():
        subj = row['subject']
        base = baseline_dict.get(subj, np.nan)
        delta = row['meanHR'] - base if not np.isnan(base) else np.nan
        delta_list.append(delta)
    df_hr['Delta_HR'] = delta_list
    print("ΔHR computed and added to DataFrame.")
    return df_hr

#### Feature HRV

In [18]:
def Compute_HRV(df_rpeaks):
    hrv_list = []
    for idx, row in df_rpeaks.iterrows():
        RR_intervals_ms = row.get('RR_intervals_ms', np.array([]))
        if len(RR_intervals_ms) > 1:
            meanRR = np.mean(RR_intervals_ms)
            SDNN   = np.std(RR_intervals_ms, ddof=1)
            RMSSD  = np.sqrt(np.mean(np.diff(RR_intervals_ms)**2))
            pNN50  = np.sum(np.diff(RR_intervals_ms) > 50.0) / len(RR_intervals_ms) * 100
            meanHRV = 60000.0 / meanRR  # bpm
            HRV_features = {
                'meanRR_ms': meanRR,
                'SDNN_ms': SDNN,
                'RMSSD_ms': RMSSD,
                'pNN50': pNN50,
                'meanHRV_bpm': meanHRV
            }
        else:
            HRV_features = {
                'meanRR_ms': np.nan,
                'SDNN_ms': np.nan,
                'RMSSD_ms': np.nan,
                'pNN50': np.nan,
                'meanHRV_bpm': np.nan
            }
        hrv_list.append(HRV_features)

    df_hrv_features = pd.DataFrame(hrv_list)
    df_hrv = pd.concat([df_rpeaks.reset_index(drop=True), df_hrv_features], axis=1)
    print(f"HRV features computed for {len(df_hrv)} windows using ms standard.")
    return df_hrv

In [19]:
def Add_Missing_ECG_Features(df):
    extra_feats = []
    for idx, row in df.iterrows():
        signal = np.array(row['signal'], dtype=float)
        R_peaks = np.array(row['R_peaks'], dtype=int)
        
        mean_ecg = float(np.mean(signal)) if 'mean_ecg' not in row else row['mean_ecg']
        std_ecg  = float(np.std(signal)) if 'std_ecg' not in row else row['std_ecg']
        rms_ecg  = float(np.sqrt(np.mean(signal**2))) if 'rms_ecg' not in row else row['rms_ecg']
        ptp_ecg  = float(np.ptp(signal)) if 'ptp_ecg' not in row else row['ptp_ecg']
        
        fs = 1 / np.mean(np.diff(row['times'])) if len(row['times']) > 1 else 1.0
        freqs, psd = welch(signal, fs=fs, nperseg=min(1024, len(signal)))
        total_power = float(np.sum(psd)) if 'total_power_ecg' not in row else row['total_power_ecg']
        band_idx = (freqs >= 0.5) & (freqs <= 3)
        bp_0_5_3 = float(np.trapz(psd[band_idx], freqs[band_idx])) if np.any(band_idx) and 'bp_0_5_3_ecg' not in row else row.get('bp_0_5_3_ecg', 0.0)
        
        n_beats = len(R_peaks) if 'n_beats' not in row else row['n_beats']
        
        extra_feats.append({
            'mean_ecg': mean_ecg,
            'std_ecg': std_ecg,
            'rms_ecg': rms_ecg,
            'ptp_ecg': ptp_ecg,
            'total_power_ecg': total_power,
            'bp_0_5_3_ecg': bp_0_5_3,
            'n_beats': n_beats
        })
    
    df_extra = pd.DataFrame(extra_feats)
    df_combined = pd.concat([df.reset_index(drop=True), df_extra], axis=1)
    print(f"Added missing ECG features for {len(df_combined)} windows.")
    return df_combined

#### Apply

In [20]:
def Features(df_cleaned, pre_ictal_only_baseline=False, show_rpeaks=False):
    
    print("\nDetecting R-peaks and computing HR series...")
    df_rpeaks = Apply_R_peak_Detection(df_cleaned, show_plot=show_rpeaks)
    
    print("\nComputing HR summary features...")
    df_hr = Compute_HR_summary(df_rpeaks)
    
    print("\nComputing baseline HR & ΔHR...")
    baseline_dict = Compute_Baseline_HR(df_hr, pre_ictal_only=pre_ictal_only_baseline)
    df_delta_hr = Compute_Delta_HR(df_hr, baseline_dict)
    
    print("\nComputing HRV features (all in ms)...")
    df_hrv = Compute_HRV(df_delta_hr)
    
    print("\nAdding ECG amplitude & frequency features...")
    df_final = Add_Missing_ECG_Features(df_hrv)
    
    print("\nProcessing complete.")
    print("Summary of labels:")
    print(df_final['label'].value_counts())
    
    return {
        "df_rpeaks": df_rpeaks,
        "df_hr": df_hr,
        "baseline_dict": baseline_dict,
        "df_delta_hr": df_delta_hr,
        "df_hrv": df_final
    }

### Feature Extraction ACC

In [ ]:
def extract_acc_features(seg_df):

    df = seg_df.copy()

    df["activity_score"] = 0.0
    df["rms_mag"] = 0.0
    df["std_mag"] = 0.0

    for idx, row in df.iterrows():
        acc_x = np.array(row["ACC_X"], dtype=float)
        acc_y = np.array(row["ACC_Y"], dtype=float)
        acc_z = np.array(row["ACC_Z"], dtype=float)

        mag = np.sqrt(acc_x**2 + acc_y**2 + acc_z**2)

        mag_motion = row.get("mag_motion_window", mag)
        if mag_motion is None:
            mag_motion = mag

        df.at[idx, "activity_score"] = float(np.sqrt(np.mean(np.array(mag_motion)**2)))
        df.at[idx, "rms_mag"] = float(np.sqrt(np.mean(mag**2)))
        df.at[idx, "std_mag"] = float(np.std(mag))

    print(f"Extracted features for {len(df)} windows, all original columns retained.")
    return df

In [ ]:
def calculate_acc_score(acc_df):
    cols = ["acc_magnitude", "activity_score", "rms_mag", "std_mag"]

    for c in cols:
        acc_df[c] = pd.to_numeric(acc_df[c], errors="coerce")

    acc_df[cols] = acc_df[cols].fillna(0)

    acc_df["acc_score"] = (
        acc_df["acc_magnitude"] +
        acc_df["activity_score"] +
        acc_df["rms_mag"] +
        acc_df["std_mag"]
    ) / 4

    acc_df["acc_score"] = (
        acc_df["acc_score"] - acc_df["acc_score"].min()
    ) / (
        acc_df["acc_score"].max()
        - acc_df["acc_score"].min()
        + 1e-8
    )

    return acc_df

### Apply All Function

In [21]:
def Process_ECG_Subject(subject_id, 
                        base_folder=r"F:\data", 
                        participants_file=r"F:\data\participants.tsv",
                        pre=30, post=30,
                        window_sec=30, step_sec=10,
                        label_threshold=0.8,
                        pre_ictal_only_baseline=False,
                        show_rpeaks=False):


    # Read ECG
    ecg_dict = Read_ECG(subject_id, base_folder=base_folder)
    ecg_df = ECG_Dataframe(ecg_dict, subject_id, participants_file)

    # Read Annotations
    ann_dict = Read_annotation(subject_id, base_folder=base_folder)
    ann_df = Annotation_Dataframe(ann_dict, subject_id)

    # Merge ECG and annotations
    merged_df = merge_edf_events(ecg_df, ann_df)

    # Remove unwanted events
    clean_df = remove_unwanted_events(merged_df)

    # Preprocess ECG
    preprocessed_df = preprocess_ecg_df(clean_df)

    # Extend seizure times
    extended_df = extend_seizure_times(preprocessed_df, pre=pre, post=post)

    #  Segment ECG into windows
    seg_df = segment_ecg_windows(extended_df, 
                                 window_sec=window_sec, 
                                 step_sec=step_sec, 
                                 label_threshold=label_threshold,
                                 use_extended=True)

    #  Extract features
    features_dict = Features(seg_df, 
                             pre_ictal_only_baseline=pre_ictal_only_baseline, 
                             show_rpeaks=show_rpeaks)

    return clean_df, preprocessed_df, features_dict

In [ ]:
def process_subject_ACC(subject_id, base_folder, participants_file,
                        pre_extend=30, post_extend=30,
                        window_sec=60, step_sec=10, label_threshold=0.8,
                        acc_fs=25):

    acc_data = Read_ACC(subject_id, base_folder=base_folder)
    
    acc_df = ACC_Dataframe(acc_data, subject_id, participants_file)
    
    ann = Read_annotation(subject_id, base_folder=base_folder)
    df_ann = Annotation_Dataframe(ann, subject_id)
    
    merged_df = merge_edf_events(acc_df, df_ann)
    
    cleaned_df = remove_unwanted_events(merged_df)
    
    preprocessed_df = preprocess_acc_df(cleaned_df)
    
    extended_df = extend_seizure_times(preprocessed_df, pre=pre_extend, post=post_extend)
    
    acc_segments_df = segment_acc_windows(extended_df,
                                          window_sec=window_sec,
                                          step_sec=step_sec,
                                          label_threshold=label_threshold,
                                          use_extended=True)
    
    features_df = extract_acc_features(acc_segments_df)
    
    return features_df, acc_segments_df

### Vissualize    Signal

#### ECG

In [22]:
def Plot_ECG(df_before, df_after, duration=60, sfreq=256, figsize=(15, 3)):
   
    n = len(df_before)
    samples_to_show = int(duration * sfreq)

    fig, axs = plt.subplots(nrows=n, ncols=2, figsize=(figsize[0], figsize[1]*n))

    if n == 1:
        axs = [axs]

    for i, ((_, row_before), (_, row_after)) in enumerate(zip(df_before.iterrows(), df_after.iterrows())):
        signal_before = row_before['signal'][:samples_to_show]
        times_before = row_before['times'][:samples_to_show]

        signal_after = row_after['signal'][:samples_to_show]
        times_after = row_after['times'][:samples_to_show]

        axs[i][0].plot(times_before, signal_before, color='tab:red')
        axs[i][0].set_title(f"Run {row_before['run']} - Original", fontsize=11)
        axs[i][0].set_xlabel("Time (s)")
        axs[i][0].set_ylabel("ECG")
        axs[i][0].grid(True)

        axs[i][1].plot(times_after, signal_after, color='tab:green')
        axs[i][1].set_title(f"Run {row_after['run']} - Processed", fontsize=11)
        axs[i][1].set_xlabel("Time (s)")
        axs[i][1].set_ylabel("ECG")
        axs[i][1].grid(True)

    plt.tight_layout()
    plt.show()

In [23]:
def Plot_HR_Timeline(df, subject_id=None):
    
    df_sorted = df.sort_values('start_time')

    plt.figure(figsize=(14, 5))
    plt.title(f"Heart Rate Evolution Over Time {f'| {subject_id}' if subject_id else ''}", 
              fontsize=14, fontweight='bold')

    state_colors = {
        'normal': 'lightgray',
        'preictal': 'gold',
        'ictal': 'red',
        'postictal': 'orange'
    }

    plt.plot(df_sorted['start_time'], df_sorted['meanHR'], color='blue', linewidth=1.5, label='Mean HR')

    for _, row in df_sorted.iterrows():
        label = row['label'].lower()
        color = state_colors.get(label, 'white')
        start = row['start_time']
        end = row['end_time'] if 'end_time' in row else start + row.get('duration', 30)
        plt.axvspan(start, end, color=color, alpha=0.25)

    plt.xlabel("Time (s)")
    plt.ylabel("Mean HR (bpm)")
    plt.grid(alpha=0.3)
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

In [24]:
def plot_HR(df_hr, subject_id, duration=60, window_index=None):
  
    subject_num = subject_id.replace("sub-", "")
    subject_df = df_hr[df_hr['subject'] == subject_num]

    if subject_df.empty:
        print(f"No ECG data found for {subject_id}")
        return

    label_colors = {
        'normal': 'lightgray',
        'preictal': 'yellow',
        'ictal': 'red',
        'postictal': 'orange'
    }

    states_order = ['normal', 'ictal', 'preictal', 'postictal']

    for run, run_df in subject_df.groupby('run'):
        selected_rows = []

        for state in states_order:
            state_rows = run_df[run_df['label'].str.lower() == state]
            if state_rows.empty:
                continue

            if window_index is None:
                selected_rows.append(state_rows.iloc[0])
            else:
                if window_index < len(state_rows):
                    selected_rows.append(state_rows.iloc[window_index])
                else:
                    print(f"Window index {window_index} out of range for state '{state}' in Run {run}")
                    selected_rows.append(state_rows.iloc[-1])  

        if not selected_rows:
            print(f"No labeled windows found for Run {run}")
            continue

        print(f"\nRun {run}: plotting {len(selected_rows)} states side by side")

        fig, axes = plt.subplots(len(selected_rows), 2, figsize=(14, len(selected_rows) * 2.8))
        fig.suptitle(f"{subject_id} | Run {run} | ECG + HR per State",
                     fontsize=14, fontweight='bold')

        if len(selected_rows) == 1:
            axes = np.expand_dims(axes, axis=0)

        for i, row in enumerate(selected_rows):
            label = row['label'].lower()
            color = label_colors.get(label, 'white')

            signal = np.array(row['signal'])
            times = np.array(row['times'], dtype=float)
            r_peaks = np.array(row['R_peaks'])
            hr_series = np.array(row['HR_series']) if 'HR_series' in row else np.zeros(len(signal))

            sfreq = 1 / np.mean(np.diff(times))

            end_idx = int(sfreq * duration)
            signal = signal[:end_idx]
            times = times[:end_idx]

            if len(r_peaks) > 0:
                r_peaks_idx = np.searchsorted(times, r_peaks)
                r_peaks_idx = r_peaks_idx[r_peaks_idx < len(signal)]

            ax_ecg = axes[i, 0]
            ax_ecg.axvspan(times[0], times[-1], color=color, alpha=0.25)
            ax_ecg.plot(times, signal, color='blue', linewidth=1.1)
            if len(r_peaks) > 0:
                ax_ecg.scatter(times[r_peaks_idx], signal[r_peaks_idx], color='red', s=15, label='R-peaks')
            ax_ecg.set_ylabel("ECG (norm.)")
            ax_ecg.set_title(f"{label.title()} - ECG + R-peaks", fontsize=11)
            ax_ecg.grid(alpha=0.3)

            ax_hr = axes[i, 1]
            hr_times = np.linspace(times[0], times[-1], len(hr_series))
            ax_hr.axvspan(hr_times[0], hr_times[-1], color=color, alpha=0.25)
            ax_hr.plot(hr_times, hr_series, color='green', linewidth=1.3)
            ax_hr.set_ylabel("HR (bpm)")
            ax_hr.set_xlabel("Time (s)")
            ax_hr.set_title(f"{label.title()} - HR Series", fontsize=11)
            ax_hr.grid(alpha=0.3)

        plt.tight_layout()
        plt.show()

### Analysis ECG

#### Read

In [25]:
def Load_All_ECG(base_path="F:\data\Code\Features 2\csv"):
    import os, pandas as pd

    ecg_data = {}

    for file in os.listdir(base_path):
        if file.startswith("ECG") :
            key = file.replace(".csv", "")   
            ecg_data[key] = pd.read_csv(os.path.join(base_path, file))

    return ecg_data

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Abdelrhamn\AppData\Local\Temp\ipykernel_15460\4139428629.py:1: SyntaxWarning: invalid escape sequence '\d'
  def Load_All_ECG(base_path="F:\data\Code\Features 2\csv"):


In [26]:
def Run_ECG_Function(ecg_dict, ecg_name, func_name, label_col='label'):

    df = ecg_dict[ecg_name]

    if func_name == "box":
        Plot_Boxplots(df, label_col)

    elif func_name == "violin":
        Plot_Violin(df, label_col)

    elif func_name == "dist":
        Plot_Distribution(df, label_col)

    elif func_name == "hist":
        Plot_Histograms(df, label_col)

    elif func_name == "corr":
        Plot_Correlation(df)

    elif func_name == "analysis":
        Analysis_Feature(df, label_col)

    elif func_name == "stats":
        result = Full_Analysis(df, label_col)
        return result

    else:
        raise ValueError("Function must be one of: box, violin, dist, hist, corr, analysis, stats")

#### Ranges

In [27]:
def Analysis_Feature(df, label_col='label', exclude_cols=None):
    import pandas as pd

    if label_col not in df.columns:
        raise ValueError(f"'{label_col}' not found in DataFrame")

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    exclude = ['subject', 'run', 'sfreq', 'start_time', 'end_time']
    if exclude_cols:
        exclude.extend(exclude_cols)

    features = [col for col in numeric_cols if col not in exclude]
    print(f"Analyzing {len(features)} numeric features: {features}\n")

    for feature in features:
        ranges_list = []

        for label in df[label_col].unique():
            values = df[df[label_col] == label][feature].dropna()
            if len(values) == 0:
                continue
            stats = {
                'Label': label,
                'Min': values.min(),
                'Max': values.max(),
                'Mean': values.mean(),
                'Std': values.std(),
                'Median': values.median(),
                '25th_percentile': values.quantile(0.25),
                '75th_percentile': values.quantile(0.75)
            }
            ranges_list.append(stats)

        df_feature = pd.DataFrame(ranges_list)

        df_feature = df_feature.sort_values(by='Mean', ascending=False).reset_index(drop=True)

        print(f"==== Feature: {feature} ====")
        display(df_feature)

In [28]:
def Full_Analysis(df, label_col='label', exclude_cols=None):

    if label_col not in df.columns:
        raise ValueError(f"'{label_col}' not found in DataFrame")

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    exclude = ['subject', 'run', 'encoded_label', 'sfreq',
               'start_time', 'end_time']

    if exclude_cols:
        exclude.extend(exclude_cols)

    features = [col for col in numeric_cols if col not in exclude]

    results = []

    for feature in features:
        groups = [
            df[df[label_col] == lbl][feature].dropna()
            for lbl in df[label_col].unique()
        ]

        if all(len(g) > 0 for g in groups):
            stat, p = kruskal(*groups)
            results.append({
                'Feature': feature,
                'H-stat': stat,
                'p-value': p
            })
        else:
            results.append({
                'Feature': feature,
                'H-stat': np.nan,
                'p-value': np.nan
            })

    df_stats = pd.DataFrame(results).sort_values('p-value')

    return df_stats

#### VIsualize Analysis

In [29]:
def Plot_Boxplots(df, label_col='label'):
    features = [ 'meanHR', 'stdHR', 'minHR', 'maxHR', 'Delta_HR', 'meanRR_ms', 'SDNN_ms','RMSSD_ms', 
                  'pNN50', 'meanHRV_bpm', 'mean_ecg', 'std_ecg', 'rms_ecg', 'ptp_ecg', 'total_power_ecg']

    n_features = len(features)
    n_cols = 3
    n_rows = math.ceil(n_features / n_cols)

    fig, axs = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    axs = axs.flatten()

    for i, feat in enumerate(features):
        sns.boxplot(x=label_col, y=feat, data=df, ax=axs[i])
        axs[i].set_title(f'{feat}', fontsize=11)
        axs[i].set_xlabel('')
        axs[i].set_ylabel(feat)

    for j in range(i+1, len(axs)):
        fig.delaxes(axs[j])

    fig.suptitle("Boxplots of Features", fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

In [30]:
def Plot_Violin(df, label_col='label', bins=30):

    features = [ 'meanHR', 'stdHR', 'minHR', 'maxHR', 'Delta_HR', 'meanRR_ms', 'SDNN_ms','RMSSD_ms', 
                  'pNN50', 'meanHRV_bpm', 'mean_ecg', 'std_ecg', 'rms_ecg', 'ptp_ecg', 'total_power_ecg']

    n_features = len(features)
    n_cols = 3
    n_rows = math.ceil(n_features / n_cols)

    fig, axs = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    axs = axs.flatten()

    for i, feat in enumerate(features):
        sns.violinplot(x=label_col, y=feat, data=df, ax=axs[i], inner='quartile')
        axs[i].set_title(f'Violin of {feat}', fontsize=11)
        axs[i].set_xlabel(feat)
        axs[i].set_ylabel('Count')

    for j in range(i+1, len(axs)):
        fig.delaxes(axs[j])

    fig.suptitle("Violines of Features", fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

In [31]:
def Plot_Distribution(df, label_col='label'):
    features = [ 'meanHR', 'stdHR', 'minHR', 'maxHR', 'Delta_HR', 'meanRR_ms', 'SDNN_ms','RMSSD_ms', 
                  'pNN50', 'meanHRV_bpm', 'mean_ecg', 'std_ecg', 'rms_ecg', 'ptp_ecg', 'total_power_ecg']

    n_features = len(features)
    n_cols = 3
    n_rows = math.ceil(n_features / n_cols)

    fig, axs = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    axs = axs.flatten()

    for i, feat in enumerate(features):
        sns.kdeplot(data=df, x=feat, hue=label_col, fill=True, ax=axs[i])
        axs[i].set_title(f'{feat}', fontsize=11)
        axs[i].set_xlabel(feat)
        axs[i].set_ylabel('Density')

    for j in range(i+1, len(axs)):
        fig.delaxes(axs[j])

    fig.suptitle("Distributions of Features", fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

In [32]:
def Plot_Histograms(df, label_col='label', bins=30):

    features = [ 'meanHR', 'stdHR', 'minHR', 'maxHR', 'Delta_HR', 'meanRR_ms', 'SDNN_ms','RMSSD_ms', 
                  'pNN50', 'meanHRV_bpm', 'mean_ecg', 'std_ecg', 'rms_ecg', 'ptp_ecg', 'total_power_ecg']

    n_features = len(features)
    n_cols = 3
    n_rows = math.ceil(n_features / n_cols)

    fig, axs = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    axs = axs.flatten()

    for i, feat in enumerate(features):
        sns.histplot(data=df, x=feat, hue=label_col, kde=False, bins=bins, ax=axs[i], alpha=0.7)
        axs[i].set_title(f'Histogram of {feat}', fontsize=11)
        axs[i].set_xlabel(feat)
        axs[i].set_ylabel('Count')

    for j in range(i+1, len(axs)):
        fig.delaxes(axs[j])

    fig.suptitle("Histograms of Features", fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

In [33]:
def Plot_Correlation(df):

    num_cols = [ 'meanHR', 'stdHR', 'minHR', 'maxHR', 'Delta_HR', 'meanRR_ms', 'SDNN_ms','RMSSD_ms', 
                  'pNN50', 'meanHRV_bpm', 'mean_ecg', 'std_ecg', 'rms_ecg', 'ptp_ecg', 'total_power_ecg']

    corr = df[num_cols].corr()
    plt.figure(figsize=(10,8))
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
    plt.title("Correlation Matrix of Features")
    plt.tight_layout()
    plt.show()

### Analysis ACC

#### Read

In [ ]:
def Load_All_ACC(base_path="F:\\data\\Code\\Features 2\\acc\\ACC Final\\features"):

    acc_data = {}

    for file in os.listdir(base_path):
        if file.startswith("ACC") :
            key = file.replace(".csv", "")   
            acc_data[key] = pd.read_csv(os.path.join(base_path, file))

    return acc_data

In [ ]:
def Run_ACC_Function(acc_dict, acc_name, func_name, label_col='label'):

    df = acc_dict[acc_name]

    if func_name == "box":
        Plot_Boxplots(df, label_col)

    elif func_name == "violin":
        Plot_Violin(df, label_col)

    elif func_name == "dist":
        Plot_Distribution(df, label_col)

    elif func_name == "hist":
        Plot_Histograms(df, label_col)

    elif func_name == "corr":
        Plot_Correlation(df)

    elif func_name == "analysis":
        Analysis_Feature(df, label_col)

    elif func_name == "stats":
        result = Full_Analysis(df, label_col)
        return result

    else:
        raise ValueError("Function must be one of: box, violin, dist, hist, corr, analysis, stats")

#### Ranges

In [ ]:
def Analysis_Feature(df, label_col='label', exclude_cols=None):

    if label_col not in df.columns:
        raise ValueError(f"'{label_col}' not found in DataFrame")

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    exclude = ['subject', 'run', 'window_id', 'start_time', 'end_time','preictal_ratio', 'ictal_ratio', 'postictal_ratio', 'window_length', 'sfreq',]
    if exclude_cols:
        exclude.extend(exclude_cols)

    features = [col for col in numeric_cols if col not in exclude]
    print(f"Analyzing {len(features)} numeric features: {features}\n")

    for feature in features:
        ranges_list = []

        for label in df[label_col].unique():
            values = df[df[label_col] == label][feature].dropna()
            if len(values) == 0:
                continue
            stats = {
                'Label': label,
                'Min': values.min(),
                'Max': values.max(),
                'Mean': values.mean(),
                'Std': values.std(),
                'Median': values.median(),
                '25th_percentile': values.quantile(0.25),
                '75th_percentile': values.quantile(0.75)
            }
            ranges_list.append(stats)

        df_feature = pd.DataFrame(ranges_list)

        df_feature = df_feature.sort_values(by='Mean', ascending=False).reset_index(drop=True)

        print(f"==== Feature: {feature} ====")
        display(df_feature)

In [ ]:
def Full_Analysis(df, label_col='label', exclude_cols=None):

    if label_col not in df.columns:
        raise ValueError(f"'{label_col}' not found in DataFrame")

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    exclude = ['subject', 'run', 'encoded_label', 'sfreq','window_id',
               'start_time', 'end_time','preictal_ratio', 'ictal_ratio', 'postictal_ratio', 'window_length', 'sfreq',]

    if exclude_cols:
        exclude.extend(exclude_cols)

    features = [col for col in numeric_cols if col not in exclude]

    results = []

    for feature in features:
        groups = [
            df[df[label_col] == lbl][feature].dropna()
            for lbl in df[label_col].unique()
        ]

        if all(len(g) > 0 for g in groups):
            stat, p = kruskal(*groups)
            results.append({
                'Feature': feature,
                'H-stat': stat,
                'p-value': p
            })
        else:
            results.append({
                'Feature': feature,
                'H-stat': np.nan,
                'p-value': np.nan
            })

    df_stats = pd.DataFrame(results).sort_values('p-value')

    return df_stats

#### VIsualize Analysis

In [ ]:
def Plot_Boxplots(df, label_col='label'):
    features = [ 'activity_score', 'rms_mag', 'std_mag']

    n_features = len(features)
    n_cols = 3
    n_rows = math.ceil(n_features / n_cols)

    fig, axs = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    axs = axs.flatten()

    for i, feat in enumerate(features):
        sns.boxplot(x=label_col, y=feat, data=df, ax=axs[i])
        axs[i].set_title(f'{feat}', fontsize=11)
        axs[i].set_xlabel('')
        axs[i].set_ylabel(feat)

    for j in range(i+1, len(axs)):
        fig.delaxes(axs[j])

    fig.suptitle("Boxplots of Features", fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

In [ ]:
def Plot_Violin(df, label_col='label', bins=30):

    features = [ 'activity_score', 'rms_mag', 'std_mag']


    n_features = len(features)
    n_cols = 3
    n_rows = math.ceil(n_features / n_cols)

    fig, axs = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    axs = axs.flatten()

    for i, feat in enumerate(features):
        sns.violinplot(x=label_col, y=feat, data=df, ax=axs[i], inner='quartile')
        axs[i].set_title(f'Violin of {feat}', fontsize=11)
        axs[i].set_xlabel(feat)
        axs[i].set_ylabel('Count')

    for j in range(i+1, len(axs)):
        fig.delaxes(axs[j])

    fig.suptitle("Violines of Features", fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

In [ ]:
def Plot_Distribution(df, label_col='label'):
    features = ['activity_score', 'rms_mag', 'std_mag']

    n_features = len(features)
    n_cols = 3
    n_rows = math.ceil(n_features / n_cols)

    fig, axs = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    axs = axs.flatten()

    for i, feat in enumerate(features):
        sns.kdeplot(data=df, x=feat, hue=label_col, fill=True, ax=axs[i])
        axs[i].set_title(f'{feat}', fontsize=11)
        axs[i].set_xlabel(feat)
        axs[i].set_ylabel('Density')

    for j in range(i+1, len(axs)):
        fig.delaxes(axs[j])

    fig.suptitle("Distributions of Features", fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

In [ ]:
def Plot_Histograms(df, label_col='label', bins=30):

    features = [ 'activity_score', 'rms_mag', 'std_mag']


    n_features = len(features)
    n_cols = 3
    n_rows = math.ceil(n_features / n_cols)

    fig, axs = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    axs = axs.flatten()

    for i, feat in enumerate(features):
        sns.histplot(data=df, x=feat, hue=label_col, kde=False, bins=bins, ax=axs[i], alpha=0.7)
        axs[i].set_title(f'Histogram of {feat}', fontsize=11)
        axs[i].set_xlabel(feat)
        axs[i].set_ylabel('Count')

    for j in range(i+1, len(axs)):
        fig.delaxes(axs[j])

    fig.suptitle("Histograms of Features", fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

In [ ]:
def Plot_Correlation(df):

    num_cols = ['activity_score', 'rms_mag', 'std_mag']


    corr = df[num_cols].corr()
    plt.figure(figsize=(10,8))
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
    plt.title("Correlation Matrix of Features")
    plt.tight_layout()
    plt.show()

### Convert post to normal

In [ ]:
def convert(df, label_col='label'):

    df = df.copy()
    df.loc[df[label_col] == 'postictal', label_col] = 'normal'
    return df

### Encoded

In [ ]:
def Encode_Labels(df, label_col='label'):
 
    encoder = LabelEncoder()
    df['encoded_label'] = encoder.fit_transform(df[label_col])

    print(" Multiclass encoding applied successfully!")
    print("Class mapping:")
    for cls, code in zip(encoder.classes_, encoder.transform(encoder.classes_)):
        print(f"  {cls} → {code}")

    return df

### Split

In [ ]:
def split(df, test_size=0.2, random_state=42):

    np.random.seed(random_state)
    
    subjects = df['subject'].unique()
    n_test = max(1, int(len(subjects) * test_size))
    
    test_subjects = np.random.choice(subjects, size=n_test, replace=False)
    
    df_test = df[df['subject'].isin(test_subjects)].copy()
    df_train = df[~df['subject'].isin(test_subjects)].copy()
    
    print(f"Train subjects: {df_train['subject'].nunique()}, Test subjects: {df_test['subject'].nunique()}")
    
    return df_train, df_test

### Balance

In [ ]:
def Balance(df, FEATURES, target='encoded_label', random_state=42):

    class_counts = df[target].value_counts()
    majority_class = class_counts.idxmax()

    df_majority = df[df[target] == majority_class]
    df_minority = df[df[target] != majority_class]

    n_majority_new = 3 * len(df_minority)

    df_majority_down = resample(
        df_majority,
        replace=False,
        n_samples=min(len(df_majority), n_majority_new),
        random_state=random_state
    )

    df_under = pd.concat([df_majority_down, df_minority]) \
                 .sample(frac=1, random_state=random_state) \
                 .reset_index(drop=True)

    print("After Undersampling:")
    print(df_under[target].value_counts())

    FEATURES_IN_DF = [f for f in FEATURES if f in df_under.columns]

    if not FEATURES_IN_DF:
        raise ValueError("None of the FEATURES are in the DataFrame!")

    df_under[FEATURES_IN_DF] = df_under[FEATURES_IN_DF].apply(
        pd.to_numeric, errors='coerce'
    )
    df_under = df_under.dropna(subset=FEATURES_IN_DF)

    X = df_under[FEATURES_IN_DF].values
    y = df_under[target].values

    
    smote = BorderlineSMOTE(
        random_state=random_state,
        kind="borderline-1"
    )

    X_res, y_res = smote.fit_resample(X, y)


    df_resampled = pd.DataFrame(X_res, columns=FEATURES_IN_DF)
    df_resampled[target] = y_res

    df_resampled = df_resampled.sample(
        frac=1, random_state=random_state
    ).reset_index(drop=True)

    print("After Borderline-SMOTE:")
    print(df_resampled[target].value_counts())

    return df_resampled


### Use saved Model

In [ ]:
def predict_probabilities(features_values):

    if isinstance(features_values, dict):
        X = pd.DataFrame([features_values])

    elif isinstance(features_values, list):
        X = pd.DataFrame([features_values], columns=feature_cols)

    elif isinstance(features_values, pd.DataFrame):
        X = features_values

    else:
        raise ValueError("Input must be dict, list, or DataFrame")

    probs = model.predict_proba(X)

    for i, p in enumerate(probs):
        print(f"Sample {i}:")
        print(f"Class 0 probability: {p[0]:.4f}")
        print(f"Class 1 probability: {p[1]:.4f}")
        print(f"Class 2 probability: {p[2]:.4f}")
        print("-" * 30)

    return probs

In [ ]:
def predict_fusion(ecg_df, acc_df, fusion_model):

    X_fusion = np.hstack([
        ecg_df[["p_class0", "p_class1", "p_class2"]].values,
        acc_df[["acc_score"]].values
    ])

    y_pred = fusion_model.predict(X_fusion)
    y_prob = fusion_model.predict_proba(X_fusion)

    result = pd.DataFrame({
        "predicted_label": y_pred,
        "prob_class0": y_prob[:, 0],
        "prob_class1": y_prob[:, 1],
        "prob_class2": y_prob[:, 2]
    })

    return result